In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon

OUTPUT_DIR = "outputs/final_analysis"
FIG_DIR = os.path.join(OUTPUT_DIR, "final_figures")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
PATHS = {
    "densenet_metrics": "outputs/metrics/densenet121_test_metrics.csv",

    "surrogate_classification": "outputs/surrogates/metrics/surrogate_classification_fidelity_metrics.csv",
    "surrogate_probability": "outputs/surrogates/metrics/surrogate_probability_fidelity_metrics.csv",

    "faithfulness_results": "outputs/faithfulness/faithfulness_results.csv",
    "faithfulness_summary": "outputs/faithfulness/faithfulness_summary.csv",

    "stability_results": "outputs/stability/stability_results.csv",
    "stability_summary": "outputs/stability/stability_summary.csv",

    "lung_relevance_results": "outputs/lung_relevance/lung_relevance_results.csv",
    "lung_relevance_summary": "outputs/lung_relevance/lung_relevance_summary.csv"
}

def safe_read_csv(path):
    if os.path.exists(path):
        print(f"Loaded: {path}")
        return pd.read_csv(path)
    else:
        print(f"Missing: {path}")
        return None

# Load all result files
densenet_df = safe_read_csv(PATHS["densenet_metrics"])
surrogate_clf_df = safe_read_csv(PATHS["surrogate_classification"])
surrogate_prob_df = safe_read_csv(PATHS["surrogate_probability"])
faithfulness_results_df = safe_read_csv(PATHS["faithfulness_results"])
faithfulness_summary_df = safe_read_csv(PATHS["faithfulness_summary"])
stability_results_df = safe_read_csv(PATHS["stability_results"])
stability_summary_df = safe_read_csv(PATHS["stability_summary"])
lung_results_df = safe_read_csv(PATHS["lung_relevance_results"])
lung_summary_df = safe_read_csv(PATHS["lung_relevance_summary"])

# DenseNet121 baseline summary
if densenet_df is not None:
    densenet_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_densenet121_baseline_summary.csv"),
        index=False
    )

    print("\nDenseNet121 baseline metrics:")
    print(densenet_df)

    metric_cols = [
        col for col in [
            "accuracy",
            "precision",
            "sensitivity_recall",
            "specificity",
            "f1_score",
            "roc_auc"
        ]
        if col in densenet_df.columns
    ]

    if len(metric_cols) > 0:
        values = densenet_df.iloc[0][metric_cols].astype(float)

        plt.figure(figsize=(9, 5))
        plt.bar(metric_cols, values)
        plt.ylim(0, 1)
        plt.xticks(rotation=30, ha="right")
        plt.ylabel("Score")
        plt.title("DenseNet121 Test Performance")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "densenet121_test_performance.png"), dpi=300)
        plt.close()

Loaded: outputs/metrics/densenet121_test_metrics.csv
Loaded: outputs/surrogates/metrics/surrogate_classification_fidelity_metrics.csv
Loaded: outputs/surrogates/metrics/surrogate_probability_fidelity_metrics.csv
Loaded: outputs/faithfulness/faithfulness_results.csv
Loaded: outputs/faithfulness/faithfulness_summary.csv
Loaded: outputs/stability/stability_results.csv
Loaded: outputs/stability/stability_summary.csv
Loaded: outputs/lung_relevance/lung_relevance_results.csv
Loaded: outputs/lung_relevance/lung_relevance_summary.csv

DenseNet121 baseline metrics:
   accuracy  precision  sensitivity_recall  specificity  f1_score   roc_auc  \
0       0.9   0.927273            0.864407     0.934426  0.894737  0.948041   

   true_negative  false_positive  false_negative  true_positive  
0             57               4               8             51  


In [2]:
# Surrogate classification fidelity summary
if surrogate_clf_df is not None:
    # Remove Random Forest if present, since you skipped it or do not want it in final comparison
    surrogate_clf_df = surrogate_clf_df[
        ~surrogate_clf_df["model"].str.contains("random_forest", case=False, na=False)
    ].reset_index(drop=True)

    surrogate_clf_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_surrogate_classification_fidelity.csv"),
        index=False
    )

    print("\nSurrogate classification fidelity:")
    print(surrogate_clf_df)

    if "fidelity_accuracy" in surrogate_clf_df.columns:
        plt.figure(figsize=(8, 5))
        plt.bar(
            surrogate_clf_df["model"],
            surrogate_clf_df["fidelity_accuracy"]
        )
        plt.ylim(0, 1)
        plt.ylabel("Fidelity accuracy")
        plt.title("Global Surrogate Fidelity to DenseNet121 Predictions")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "surrogate_classification_fidelity.png"), dpi=300)
        plt.close()


Surrogate classification fidelity:
                 model  fidelity_accuracy  fidelity_precision  \
0  logistic_regression           0.983333            0.964912   
1        decision_tree           0.925000            0.942308   

   fidelity_recall  fidelity_f1  fidelity_auc  
0         1.000000     0.982143      0.997203  
1         0.890909     0.915888      0.921958  


In [3]:
# Surrogate probability fidelity summary
if surrogate_prob_df is not None:
    surrogate_prob_df = surrogate_prob_df[
        ~surrogate_prob_df["model"].str.contains("random_forest", case=False, na=False)
    ].reset_index(drop=True)

    surrogate_prob_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_surrogate_probability_fidelity.csv"),
        index=False
    )

    print("\nSurrogate probability fidelity:")
    print(surrogate_prob_df)

    if "probability_mae" in surrogate_prob_df.columns:
        plt.figure(figsize=(8, 5))
        plt.bar(
            surrogate_prob_df["model"],
            surrogate_prob_df["probability_mae"]
        )
        plt.ylabel("MAE against DenseNet121 TB probability")
        plt.title("Surrogate Probability Imitation Error")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "surrogate_probability_mae.png"), dpi=300)
        plt.close()

    if "probability_correlation" in surrogate_prob_df.columns:
        plt.figure(figsize=(8, 5))
        plt.bar(
            surrogate_prob_df["model"],
            surrogate_prob_df["probability_correlation"]
        )
        plt.ylim(-1, 1)
        plt.ylabel("Correlation with DenseNet121 TB probability")
        plt.title("Surrogate Probability Agreement")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "surrogate_probability_correlation.png"), dpi=300)
        plt.close()


Surrogate probability fidelity:
                     model  probability_mae  probability_mse  \
0         ridge_regression         0.112914         0.026461   
1  decision_tree_regressor         0.095751         0.036968   

   probability_rmse  probability_r2  probability_correlation  
0          0.162669        0.859287                 0.934370  
1          0.192270        0.803415                 0.913995  


In [4]:
# Faithfulness statistical tests
faithfulness_stats = []

if faithfulness_results_df is not None:
    for percentage in sorted(faithfulness_results_df["mask_percentage"].unique()):
        subset = faithfulness_results_df[
            faithfulness_results_df["mask_percentage"] == percentage
        ]

        pivot = subset.pivot_table(
            index="image_id",
            columns="mask_type",
            values="probability_drop",
            aggfunc="mean"
        ).dropna()

        if {"high", "low"}.issubset(pivot.columns):
            stat, p = wilcoxon(pivot["high"], pivot["low"])
            faithfulness_stats.append({
                "mask_percentage": percentage,
                "comparison": "high_vs_low",
                "n_images": len(pivot),
                "mean_high_drop": pivot["high"].mean(),
                "mean_low_drop": pivot["low"].mean(),
                "mean_difference": (pivot["high"] - pivot["low"]).mean(),
                "wilcoxon_statistic": stat,
                "p_value": p
            })

        if {"high", "random"}.issubset(pivot.columns):
            stat, p = wilcoxon(pivot["high"], pivot["random"])
            faithfulness_stats.append({
                "mask_percentage": percentage,
                "comparison": "high_vs_random",
                "n_images": len(pivot),
                "mean_high_drop": pivot["high"].mean(),
                "mean_random_drop": pivot["random"].mean(),
                "mean_difference": (pivot["high"] - pivot["random"]).mean(),
                "wilcoxon_statistic": stat,
                "p_value": p
            })

    faithfulness_stats_df = pd.DataFrame(faithfulness_stats)
    faithfulness_stats_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_faithfulness_wilcoxon_tests.csv"),
        index=False
    )

    print("\nFaithfulness Wilcoxon tests:")
    print(faithfulness_stats_df)

    if faithfulness_summary_df is not None:
        faithfulness_summary_df.to_csv(
            os.path.join(OUTPUT_DIR, "final_faithfulness_summary.csv"),
            index=False
        )

        plt.figure(figsize=(9, 5))

        for mode in ["high", "low", "random"]:
            subset = faithfulness_summary_df[
                faithfulness_summary_df["mask_type"] == mode
            ]

            if len(subset) > 0:
                plt.plot(
                    subset["mask_percentage"],
                    subset["mean_probability_drop"],
                    marker="o",
                    label=mode
                )

        plt.xlabel("Masked area (%)")
        plt.ylabel("Mean TB probability drop")
        plt.title("Faithfulness: Probability Drop After SHAP-Based Occlusion")
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "faithfulness_probability_drop.png"), dpi=300)
        plt.close()


Faithfulness Wilcoxon tests:
   mask_percentage      comparison  n_images  mean_high_drop  mean_low_drop  \
0                5     high_vs_low        45       -0.443669      -0.016496   
1                5  high_vs_random        45       -0.443669            NaN   
2               10     high_vs_low        45       -0.415489      -0.136930   
3               10  high_vs_random        45       -0.415489            NaN   
4               15     high_vs_low        45       -0.366047      -0.223497   
5               15  high_vs_random        45       -0.366047            NaN   
6               20     high_vs_low        45       -0.324263      -0.266998   
7               20  high_vs_random        45       -0.324263            NaN   
8               30     high_vs_low        45       -0.253148      -0.355575   
9               30  high_vs_random        45       -0.253148            NaN   

   mean_difference  wilcoxon_statistic       p_value  mean_random_drop  
0        -0.427173         

In [5]:
# Stability summary
if stability_summary_df is not None:
    stability_summary_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_stability_summary.csv"),
        index=False
    )

    print("\nStability summary:")
    print(stability_summary_df)

    metrics_to_plot = [
        "mean_cosine_similarity",
        "mean_spearman_correlation",
        "mean_jaccard_top10"
    ]

    for metric in metrics_to_plot:
        if metric in stability_summary_df.columns:
            plt.figure(figsize=(10, 5))
            plt.bar(
                stability_summary_df["perturbation"],
                stability_summary_df[metric]
            )
            plt.ylim(0, 1)
            plt.xticks(rotation=30, ha="right")
            plt.ylabel(metric.replace("_", " "))
            plt.title(f"SHAP Stability: {metric.replace('_', ' ')}")
            plt.tight_layout()
            plt.savefig(os.path.join(FIG_DIR, f"stability_{metric}.png"), dpi=300)
            plt.close()

    if "mean_absolute_probability_change" in stability_summary_df.columns:
        plt.figure(figsize=(10, 5))
        plt.bar(
            stability_summary_df["perturbation"],
            stability_summary_df["mean_absolute_probability_change"]
        )
        plt.xticks(rotation=30, ha="right")
        plt.ylabel("Mean absolute TB probability change")
        plt.title("DenseNet121 Prediction Change Under Perturbations")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "stability_probability_change.png"), dpi=300)
        plt.close()


Stability summary:
       perturbation  mean_cosine_similarity  std_cosine_similarity  \
0  brightness_minus                0.828913               0.037062   
1   brightness_plus                0.839827               0.025498   
2    contrast_minus                0.881339               0.020345   
3     contrast_plus                0.880694               0.019864   
4    gaussian_noise                0.721419               0.012907   
5          rotation                0.709770               0.024593   

   mean_spearman_correlation  std_spearman_correlation  mean_jaccard_top10  \
0                   0.607839                  0.087275            0.353663   
1                   0.634763                  0.062908            0.367050   
2                   0.719809                  0.044939            0.438808   
3                   0.714131                  0.039638            0.434866   
4                   0.554027                  0.051665            0.227795   
5                   0

In [6]:
# Lung-region relevance summary
lung_stats = []

if lung_results_df is not None and len(lung_results_df) > 0:
    lung_results_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_lung_relevance_results.csv"),
        index=False
    )

    print("\nLung relevance results preview:")
    print(lung_results_df.head())

    # Test whether lung enrichment is greater than 1
    if "lung_enrichment" in lung_results_df.columns:
        enrichment = lung_results_df["lung_enrichment"].dropna()

        if len(enrichment) > 0:
            # One-sample Wilcoxon against 1.0
            stat, p_two_sided = wilcoxon(enrichment - 1.0)

            lung_stats.append({
                "test": "lung_enrichment_vs_1",
                "n_images": len(enrichment),
                "mean_lung_enrichment": enrichment.mean(),
                "median_lung_enrichment": enrichment.median(),
                "wilcoxon_statistic": stat,
                "p_value_two_sided": p_two_sided
            })

    lung_stats_df = pd.DataFrame(lung_stats)
    lung_stats_df.to_csv(
        os.path.join(OUTPUT_DIR, "final_lung_relevance_statistical_tests.csv"),
        index=False
    )

    print("\nLung relevance statistical tests:")
    print(lung_stats_df)

    if lung_summary_df is not None:
        lung_summary_df.to_csv(
            os.path.join(OUTPUT_DIR, "final_lung_relevance_summary.csv"),
            index=False
        )

    if {
        "lung_area_fraction",
        "lung_attribution_ratio"
    }.issubset(lung_results_df.columns):
        plt.figure(figsize=(6, 6))
        plt.scatter(
            lung_results_df["lung_area_fraction"],
            lung_results_df["lung_attribution_ratio"],
            alpha=0.75
        )

        min_val = min(
            lung_results_df["lung_area_fraction"].min(),
            lung_results_df["lung_attribution_ratio"].min()
        )
        max_val = max(
            lung_results_df["lung_area_fraction"].max(),
            lung_results_df["lung_attribution_ratio"].max()
        )

        plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
        plt.xlabel("Lung area fraction")
        plt.ylabel("Lung attribution ratio")
        plt.title("SHAP Attribution Inside Lung Region")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "lung_attribution_vs_area.png"), dpi=300)
        plt.close()

    if "lung_attribution_ratio" in lung_results_df.columns:
        plt.figure(figsize=(8, 5))
        plt.hist(lung_results_df["lung_attribution_ratio"], bins=15, edgecolor="black")
        plt.xlabel("Lung attribution ratio")
        plt.ylabel("Number of images")
        plt.title("Distribution of SHAP Attribution Inside Lung Region")
        plt.tight_layout()
        plt.savefig(os.path.join(FIG_DIR, "lung_attribution_ratio_distribution.png"), dpi=300)
        plt.close()


Lung relevance results preview:
        image_id                                         image_path  \
0  MCUCXR_0255_1  /user/HS401/mn01409/Documents/Dissertation/dat...   
1  MCUCXR_0309_1  /user/HS401/mn01409/Documents/Dissertation/dat...   
2  MCUCXR_0173_1  /user/HS401/mn01409/Documents/Dissertation/dat...   
3  MCUCXR_0059_0  /user/HS401/mn01409/Documents/Dissertation/dat...   
4  MCUCXR_0150_1  /user/HS401/mn01409/Documents/Dissertation/dat...   

   true_label  cnn_prediction  cnn_probability_tb case_type  \
0           1               1            0.999776        TP   
1           1               1            0.999611        TP   
2           1               1            0.994467        TP   
3           0               0            0.049811        TN   
4           1               1            0.995904        TP   

                                      left_mask_path  \
0  data/Montgomery/ManualMask/leftMask/MCUCXR_025...   
1  data/Montgomery/ManualMask/leftMask/MCUCXR_030

In [9]:
# Create final text summary
summary_lines = []

summary_lines.append("Stage 8 Final Analysis Summary")
summary_lines.append("=" * 40)
summary_lines.append("")

if densenet_df is not None:
    summary_lines.append("DenseNet121 baseline metrics:")
    summary_lines.append(densenet_df.to_string(index=False))
    summary_lines.append("")

if surrogate_clf_df is not None:
    summary_lines.append("Surrogate classification fidelity:")
    summary_lines.append(surrogate_clf_df.to_string(index=False))
    summary_lines.append("")

if surrogate_prob_df is not None:
    summary_lines.append("Surrogate probability fidelity:")
    summary_lines.append(surrogate_prob_df.to_string(index=False))
    summary_lines.append("")

if faithfulness_summary_df is not None:
    summary_lines.append("Faithfulness summary:")
    summary_lines.append(faithfulness_summary_df.to_string(index=False))
    summary_lines.append("")

if stability_summary_df is not None:
    summary_lines.append("Stability summary:")
    summary_lines.append(stability_summary_df.to_string(index=False))
    summary_lines.append("")

if lung_summary_df is not None:
    summary_lines.append("Lung-region relevance summary:")
    summary_lines.append(lung_summary_df.to_string(index=False))
    summary_lines.append("")

summary_path = os.path.join(OUTPUT_DIR, "final_discussion_notes.txt")

with open(summary_path, "w") as f:
    f.write("\n".join(summary_lines))

print(f"Saved final discussion notes to {summary_path}")

Saved final discussion notes to outputs/final_analysis/final_discussion_notes.txt
